# Bloco de Transformer Encoder

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

Um bloco de encoder Transformer é *Self-Attention → Add & Norm → Feed-Forward → Add & Norm*. Empilhar $N$ desses transforma uma sequência de embeddings de token em embeddings contextualizados, que podem ser agregados para classificação, marcação de sequência, etc.


## Formulação Matemática

$$\begin{aligned}
z &= \text{LN}(x + \text{MHA}(x))\\
y &= \text{LN}(z + \text{FFN}(z))\end{aligned}$$

$$\text{FFN}(x) = \text{GELU}(xW_1 + b_1) W_2 + b_2$$

Conexões residuais preservam o fluxo do gradiente; o LayerNorm mantém as ativações em uma escala estável.


## Implementação


In [ ]:
import math, torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.ln2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, _ = self.mha(x, x, x, key_padding_mask=mask, need_weights=False)
        x = self.ln1(x + self.drop(attn_out))
        x = self.ln2(x + self.drop(self.ff(x)))
        return x

class MiniEncoder(nn.Module):
    def __init__(self, vocab, d_model=128, n_heads=4, d_ff=512, n_layers=2, n_classes=2, max_len=64):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        self.layers = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.cls = nn.Linear(d_model, n_classes)

    def forward(self, ids, mask=None):
        pos = torch.arange(ids.size(1), device=ids.device)
        x = self.embed(ids) + self.pos(pos)
        for layer in self.layers:
            x = layer(x, mask=mask)
        return self.cls(x.mean(dim=1))


## Experimento


In [ ]:
model = MiniEncoder(vocab=1000)
ids = torch.randint(0, 1000, (4, 32))
logits = model(ids)
print('logits:', logits.shape)
print('params:', sum(p.numel() for p in model.parameters()))


## Discussão

- A máscara de padding é essencial quando o batch tem sequências de tamanho variável.
- A posição do LayerNorm importa: post-norm (usado aqui) é a escolha clássica do Vaswani; pre-norm é mais comum em código moderno e mais fácil de treinar.
- A razão de expansão do FFN (4× por padrão) concentra a maior parte dos parâmetros do modelo.


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
